In [84]:
import pandas as pd
import warnings
from statistics import mean
from sklearn.neighbors import KDTree
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore')

In [85]:
data= pd.read_csv("feedback_aivancity_tinycoaching - feedback_v2.csv").drop(columns=["Age","Genre"])
data.head()

,id user,id formation,Nom formation,id domaine,ref domaine,nom domaine,id compétence,ref compétence,nom compétence,id theme,...,nom contenu,type contenu article,type contenu video,type contenu audio,type contenu rich media,durée contenu en s,Contenu déjà proposé,Contenu validé,Date réalisation contenu,Feedback
0,469,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,1,...,Toujours plus beau,X,NaN,NaN,NaN,180.0,0,1,2020-02-18 09:15:34,Bien
1,1170,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,1,...,La France va donner l'Alsace ?!,NaN,X,NaN,NaN,167.0,0,1,2020-05-13 18:44:05,Bien
2,1158,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,1,...,La France va donner l'Alsace ?!,NaN,X,NaN,NaN,167.0,0,0,2020-06-27 14:45:04,Bien
3,1176,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,2,...,De clic en clic,NaN,NaN,NaN,X,150.0,0,1,2020-05-07 12:54:45,Trop facile
4,1677,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,2,...,De clic en clic,NaN,NaN,NaN,X,150.0,0,1,2020-06-22 11:41:19,Trop facile


In [86]:
# on remplace les espaces dans les noms des colonnes avec un '_'
for column in list(data.columns):
    new_column = column.replace(" ","_")
    data=data.rename(columns={column:new_column})  
data.columns

Index(['id_user', 'id_formation', 'Nom_formation', 'id_domaine', 'ref_domaine',
       'nom_domaine', 'id_compétence', 'ref_compétence', 'nom_compétence',
       'id_theme', 'ref_theme', 'nom_theme', 'id_acquis', 'ref_acquis',
       'nom_acquis', 'Niveau_contenu', 'id_contenu', 'ref_contenu',
       'nom_contenu', 'type_contenu_article', 'type_contenu_video',
       'type_contenu_audio', 'type_contenu_rich_media', 'durée_contenu_en_s',
       'Contenu_déjà_proposé', 'Contenu_validé', 'Date_réalisation_contenu',
       'Feedback'],
      dtype='object')

## Nettoyage

In [87]:
# On remplace les X et les NaN par des 1 et des 0 dans les colonnes de type de contenu
list_type_contenu=['type_contenu_article', 'type_contenu_video','type_contenu_audio', 'type_contenu_rich_media']
for column in list_type_contenu:
    data[column]=data[column].fillna(0)
    data[column]=data[column].apply(lambda x: 1 if x=="X" else "Other" if x!="X" and x!=0 else x)

In [88]:
# On retire les valeurs manquantes
data = data.loc[data.nom_acquis.isna()==False] # 1 ligne
data = data.loc[data.durée_contenu_en_s.isna()==False] # 46 lignes

In [89]:
# On remplace la colonne de durée contenu en s par durée contenu en mn
data["durée_contenu_en_minutes"]=data.durée_contenu_en_s.apply(lambda x : round(x/60))
data = data.drop(columns=["durée_contenu_en_s"])
data.head()

,id_user,id_formation,Nom_formation,id_domaine,ref_domaine,nom_domaine,id_compétence,ref_compétence,nom_compétence,id_theme,...,nom_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,Contenu_déjà_proposé,Contenu_validé,Date_réalisation_contenu,Feedback,durée_contenu_en_minutes
0,469,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,1,...,Toujours plus beau,1,0,0,0,0,1,2020-02-18 09:15:34,Bien,3
1,1170,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,1,...,La France va donner l'Alsace ?!,0,1,0,0,0,1,2020-05-13 18:44:05,Bien,3
2,1158,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,1,...,La France va donner l'Alsace ?!,0,1,0,0,0,0,2020-06-27 14:45:04,Bien,3
3,1176,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,2,...,De clic en clic,0,0,0,1,0,1,2020-05-07 12:54:45,Trop facile,2
4,1677,1,DIGITAL,2,DM01,Information et données,5,CP01,1.1 Mener une recherche ou une veille d'inform...,2,...,De clic en clic,0,0,0,1,0,1,2020-06-22 11:41:19,Trop facile,2


In [90]:
# On retire les colonnes inutiles aux modèles
data = data.drop(columns=["id_formation","id_domaine","ref_domaine","id_compétence","ref_compétence","id_theme","ref_theme","id_acquis","ref_acquis","id_contenu","ref_contenu","Date_réalisation_contenu"])

# Système de recommandation ( en fonction du feedback et du contenu validé)

In [91]:
def filtrage(data,formation,domaine):
    """
    Permet de filtrer les données par formation puis par domaine

    Paramètres:
    ----------
        data(DataFrame) : les données
        formation(str) : nom de la formation
        domaine(str) : nom du domaine
    Output:
    ------
        filtered_data(DataFrame) : base de données comportant les données relatives à une formation et un domaine donnés
    """
    return data.loc[data["Nom_formation"]==formation].loc[data["nom_domaine"]==domaine].drop(columns=["Nom_formation","nom_domaine"])

def contenu(data):
    """
    permet de créer une colonne "Type de contenu" qui regroupe 4 colonnes différentes 'type_contenu_video','type_contenu_audio',
    'type_contenu_rich_media','type_contenu_article' et réunit le tout en une colonne dont les valeurs possibles sont video, audio, rich media et article 

    """
    contenu_list=['type_contenu_video','type_contenu_audio','type_contenu_rich_media','type_contenu_article']
    for column in contenu_list:
        if data[column] == 1:
            contenu = column.replace("type_contenu_",'')
            return contenu

In [92]:
def profile(data,user,critere):
    """
    Crée une matrice de profil utilisateur d'un utilisateur

    Paramètres
    ----------
        data(DataFrame) : Les données
        user(int) : l'id de l'utilisateur
        critere(str) : Feedback ou Contenu_Validé , critère par lequel on détermine les préférences de l'utilisateur 

    Output : 
        profil(Dataframe) : profil de l'utilisateur dont les colonnes sont : 
            	• Moyenne de la durée des contenus consommés
             	• % de type de contenus consommés ( autant de colonnes que de types de contenu)
                • Moyenne de niveau des contenus consommés 
                • A validé au moins un contenu ou a au moins un feedback positif
        
        NB: Si l'utilisateur a des préférences définies le profil utilisateur sera déterminé par ses préférences (niveau de contenu apprécié moyen, 
        durée de contenu moyen apprécié,etc) et dans le cas contraire, le profil utilisateur sera juste déterminé par les contenus consommés
    """
    
    user_data = data.query("id_user=={}".format(user))
    if critere=="Feedback":
        positive_data = user_data.loc[(user_data["Feedback"]=="Bien")|(user_data["Feedback"]=="Trop facile")]
    else:
        positive_data=user_data.query("Contenu_validé==1")
    
    if positive_data.shape[0] != 0:
        x = positive_data.drop(columns=['nom_compétence','nom_theme','nom_acquis','nom_contenu'])
    else:
        x = user_data.drop(columns=['nom_compétence','nom_theme','nom_acquis','nom_contenu'])
        
    x["Type_de_contenu"]= x.apply(lambda x : contenu(x),axis=1)
    x = x.drop(columns=['type_contenu_video','type_contenu_audio','type_contenu_rich_media','type_contenu_article'])
    
    pourcentage_type_contenu = dict(round(x["Type_de_contenu"].value_counts()/len(x.Type_de_contenu),2))
    mydict={"id_user": user, 
            "durée_contenu_en_minutes": round(x["durée_contenu_en_minutes"].mean(),2),
            "Niveau_contenu": round(x["Niveau_contenu"].mean(),2)
            }
    def find(mydict,type):
        """
        Retourne la valeur de chaque type de contenu
 
        """
        myvalue = 0
        for key,value in mydict.items():
            if type == key:
                myvalue = value
        return myvalue
    type_de_contenu = ["article","video","audio","rich_media"]
    for type in type_de_contenu:
        mydict.update({"type_contenu_{}".format(type):find(pourcentage_type_contenu,type)})
    if critere=="Feedback":    
        mydict.update({"A_bons_feedbacks" : bool("Trop facile" in list(user_data.Feedback.unique()) or "Bien" in list(user_data.Feedback.unique()))})
    else : 
        mydict.update({"A_validé_contenus" : bool(1 in list(user_data.Contenu_validé.unique()))})

    return mydict

def neighbour_profile(neighbours_data,critere):
    """
    Crée un profil type de voisin en fonction des utilisateurs similaires à un utilisateur donné

    Paramètres:
    ----------
        neighbours_data(DataFrame) : les données des voisins de notre utilisateur
        critere(str) : "A_bons_feedbacks" ou "A_validé_contenus" , critère par lequel on détermine les préférences de l'utilisateur
    
    Output : 
    ------
        neighbour(DataFrame) : profil type du voisin dont les caractéristiques sont une moyenne des caractéristiques des voisins
    """
    neighbour_profile = {}
    for column in list(neighbours_data.columns):
        if column not in ["id_user",critere]:
            neighbour_profile.update({column: round(mean( neighbours_data[column]),2)})
    return neighbour_profile

def content_features(data):
    """
    Crée une matrice contenant tous les contenus pouvant être proposés (qui sont dans la formation et le domaine choisis)
    Paramètre:
    ---------
        data(DataFrame) : les données
    
    Output :
    ------
        contenus(DataFrame) : Matrice des contenus dont les colonnes sont :
                   	• Nom Contenu
                    • Niveau Contenu
                 	• Durée Contenu
                    • Type de contenu ( autant de colonnes que de types de contenu)
    """
    return  data.drop(columns=["id_user","Feedback","Contenu_validé","Contenu_déjà_proposé","nom_compétence","nom_theme","nom_acquis"]).drop_duplicates()        

### Content-Based Filtering : Dans le cas ou l'utilisateur a des préférences définies, on lui propose directement des contenus similaires à ses préférences 

#### En fonction du contenu validé

In [93]:
# On filtre par formation et domaine
filtered_data= filtrage(data,"DIGITAL","Information et données")
# On crée une matrice des préférences de chaque utilisateur
user_profiles = pd.DataFrame([profile(filtered_data,user,"Contenu_Validé") for user in list(filtered_data.id_user.unique())])
# On met les colonnes de la matrice utilisateur dans le même ordre que les colonnes de la matrice des contenus pour éviter les erreurs de calcul
user_profiles = user_profiles[['id_user','Niveau_contenu','type_contenu_article','type_contenu_video','type_contenu_audio','type_contenu_rich_media','durée_contenu_en_minutes',"A_validé_contenus"]]
# On crée une matrice contenant chaque contenu et ses caractéristiques
content_data = content_features(filtered_data)
# On retire les contenus que l'utilisateur a déja consommés
user_id= 1170
content_data = content_data.loc[~content_data["nom_contenu"].isin(list(content_features(filtered_data.query("id_user=={}".format(user_id))["nom_contenu"])))]   

In [94]:
# On cherche les 5 contenus les plus similaires aux préférences de l'utilisateur
tree = KDTree(content_data.drop(columns=["nom_contenu"]))
user_id= 1170
user_row = user_profiles.query("id_user=={}".format(user_id))
dist,ind = tree.query(user_row.drop(columns=["id_user","A_validé_contenus"]),k=5)


In [95]:
# Notre utilisateur
user_row

,id_user,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes,A_validé_contenus
1,1170,3.67,0.67,0.33,0,0.0,2.0,True


In [96]:
# Les 5 contenus à recommander à l'utilisateur
content_data.iloc[ind.tolist()[0]]

,Niveau_contenu,nom_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes
36,4,Ne restez pas KO face à ces unités,1,0,0,0,1
24,4,Devenez un as de la recherche,0,0,0,1,2
32,4,On ne va pas y aller par quatre chemins,0,0,0,1,2
40,4,Vos données dans le Cloud,0,0,0,1,1
30,4,Zip Zap Zoup,0,0,0,1,1


### Collaborative Filtering : cas ou l'utilisateur n'a pas de préférences définies (pas de bons feedbacks ni de  contenus validés)

##### Sachant que notre utilisateur n'a pas de préférences définies, on ne peut pas lui recommander directement des contenus qu'il pourrait apprécier. Nous allons donc voir d'abord dans la base de données utilisateur ceux qui ressemblent le plus à notre utilisateur ( niveau de contenu moyen consommé similaire, durée de contenu moyen consommé similaires, type de contenus consommés similaire ) et qui ont des préférences définies (au moins un bon feedback ou un contenu validé). Après avoir trouvé ces utilisateurs, nous allons créer un profil type de voisin ayant commé caractéristiques la moyenne de celles de nos utilisateurs et nous allons recommander des contenus basés sur ses préférences 

### Recommandation en fonction du contenu validé

In [97]:
# On filtre par formation et domaine
filtered_data= filtrage(data,"DIGITAL","Information et données") 
# On crée une matrice des préférences de chaque utilisateur
user_profiles = pd.DataFrame([profile(filtered_data,user,"Contenu_validé") for user in list(filtered_data.id_user.unique())])
user_profiles.head()

,id_user,durée_contenu_en_minutes,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,A_validé_contenus
0,469,3.00,2.00,1.00,0.00,0,0.00,True
1,1170,2.00,3.67,0.67,0.33,0,0.00,True
2,1158,2.50,4.00,1.00,0.00,0,0.00,True
3,1176,1.50,1.50,0.00,0.00,0,1.00,True
4,1677,2.33,1.67,0.33,0.00,0,0.67,True


In [98]:
# On crée une matrice contenant chaque contenu et ses caractéristiques
contents = content_features(filtered_data)
# On retire les contenus que l'utilisateur a déja consommés
user_id = 60
contents = contents.loc[~contents["nom_contenu"].isin(list(content_features(filtered_data.query("id_user=={}".format(user_id))["nom_contenu"])))]   

In [99]:
# On met les colonnes de la matrice utilisateur dans le même ordre que les colonnes de la matrice des contenus pour éviter les erreurs de calcul
user_profiles = user_profiles[['id_user','Niveau_contenu',
 'type_contenu_article',
 'type_contenu_video',
 'type_contenu_audio',
 'type_contenu_rich_media',
 'durée_contenu_en_minutes',"A_validé_contenus"]]

In [100]:
# On cherche pour un utilisateur donné les utilisateurs qui sont le plus similaires à lui
tree1 = KDTree(user_profiles.query("A_validé_contenus==True").drop(columns=["id_user","A_validé_contenus"]))
user_id = 60
user_row = user_profiles.query("id_user=={}".format(user_id))
dist1,ind1 = tree1.query(user_row.drop(columns=["id_user","A_validé_contenus"]),k=3)
# Quand on filtre pour avoir les utilisateurs qui ont des préférences définies, les index de la matrice sont modifiés. Sachant que le modèle retourne 
# les index des voisins pour les identifier il faut retouver les indexs de bases pour avoir les véritables voisins
original_indices = list(user_profiles.index)
filtered_indices = list(user_profiles.query("A_validé_contenus==True").index)
mapping = {original_indices[i]:filtered_indices[i]  for i in range(min(len(filtered_indices), len(original_indices)))}
original_indices = [mapping[i] for i in ind1.tolist()[0]]

In [101]:
# Notre utilisateur
user_row

,id_user,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes,A_validé_contenus
36,60,4.0,0.0,1.0,0,0.0,2.0,False


In [102]:
# Ses voisins
voisins = user_profiles.iloc[original_indices]
voisins

,id_user,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes,A_validé_contenus
35,475,4.00,0.00,1.00,0,0.0,2.0,True
37,1221,4.00,0.00,1.00,0,0.0,2.0,True
1,1170,3.67,0.67,0.33,0,0.0,2.0,True


In [103]:
# On détermine à partir des 3 plus proches voisins un profil type de voisin
neighbour = pd.DataFrame([neighbour_profile(user_profiles.iloc[original_indices].query("id_user!={}".format(user_id)),"A_validé_contenus")])
neighbour

,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes
0,3.89,0.22,0.78,0,0.0,2.0


In [104]:
# Pour ce profil type de voisin, on cherche les contenus les plus similaires à ses préférences
from sklearn.neighbors import KDTree
tree2 = KDTree(contents.drop(columns=["nom_contenu"]))
dist2,ind2 = tree2.query(neighbour[0:1],k=5)

In [105]:
# Voici les 5 contenus que l'utilisateur avec l'id 60 pourrait apprécier 
contents.iloc[ind2.tolist()[0]]

,Niveau_contenu,nom_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes
1,4,La France va donner l'Alsace ?!,0,1,0,0,3
65,4,Supprimer sans paniquer,1,0,0,0,2
32,4,On ne va pas y aller par quatre chemins,0,0,0,1,2
24,4,Devenez un as de la recherche,0,0,0,1,2
34,4,Créez vous-même vos propres raccourcis,0,0,0,1,2


### Recommandation en fonction du feedback

In [106]:
# On filtre par formation et domaine
filtered_data= filtrage(data,"DIGITAL","Information et données") 
# On crée une matrice des préférences de chaque utilisateur
user_profiles = pd.DataFrame([profile(filtered_data,user,"Feedback") for user in list(filtered_data.id_user.unique())])
# On crée une matrice contenant chaque contenu et ses caractéristiques
contents = content_features(filtered_data)
# On retire les contenus que l'utilisateur a déja consommés
user_id = 60
contents = contents.loc[~contents["nom_contenu"].isin(list(content_features(filtered_data.query("id_user=={}".format(user_id))["nom_contenu"])))]   

In [107]:
# On met les colonnes de la matrice utilisateur dans le même ordre que les colonnes de la matrice des contenus pour éviter les erreurs de calcul
user_profiles = user_profiles[['id_user','Niveau_contenu',
 'type_contenu_article',
 'type_contenu_video',
 'type_contenu_audio',
 'type_contenu_rich_media',
 'durée_contenu_en_minutes',"A_bons_feedbacks"]]

In [108]:
# On cherche pour un utilisateur donné les utilisateurs qui sont le plus similaires à lui et qui ont des préférences définies
tree3 = KDTree(user_profiles.query("A_bons_feedbacks==True").drop(columns=["id_user","A_bons_feedbacks"]))
user_id = 60
user_row = user_profiles.query("id_user=={}".format(user_id))
dist3,ind3 = tree3.query(user_row.drop(columns=["id_user","A_bons_feedbacks"]),k=3)

In [109]:
# Quand on filtre pour avoir les utilisateurs qui ont des préférences définies, les index de la matrice sont modifiés. Sachant que le modèle retourne 
# les index des voisins pour les identifier il faut retouver les indexs de bases pour avoir les véritables voisins 
original_indices = list(user_profiles.index)
filtered_indices = list(user_profiles.query("A_bons_feedbacks==True").index)
mapping = {original_indices[i]:filtered_indices[i]  for i in range(min(len(filtered_indices), len(original_indices)))}
original_indices = [mapping[i] for i in ind3.tolist()[0]]

In [110]:
# Notre utilisateur
user_row

,id_user,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes,A_bons_feedbacks
36,60,4.0,0.0,1.0,0,0.0,2.0,False


In [111]:
# Ses voisins
voisins = user_profiles.iloc[original_indices]
voisins

,id_user,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes,A_bons_feedbacks
38,471,4.0,0.0,1.0,0,0.0,2.0,True
35,475,4.0,0.0,1.0,0,0.0,2.0,True
37,1221,4.0,0.0,1.0,0,0.0,2.0,True


In [112]:
# On détermine à partir des 3 plus proches voisins un profil type de voisin
neighbour = pd.DataFrame([neighbour_profile(user_profiles.iloc[original_indices].query("id_user!={}".format(user_id)),"A_bons_feedbacks")])
neighbour

,Niveau_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes
0,4.0,0.0,1.0,0,0.0,2.0


In [113]:
# Pour ce profil type de voisin, on cherche les contenus les plus similaires à ses préférences
from sklearn.neighbors import KDTree
tree = KDTree(contents.drop(columns=["nom_contenu"]))
dist,ind = tree.query(neighbour[0:1],k=5)

In [114]:
# Voici les 5 contenus que l'utilisateur avec l'id 60 pourrait apprécier
contents.iloc[ind.tolist()[0]]


,Niveau_contenu,nom_contenu,type_contenu_article,type_contenu_video,type_contenu_audio,type_contenu_rich_media,durée_contenu_en_minutes
1,4,La France va donner l'Alsace ?!,0,1,0,0,3
65,4,Supprimer sans paniquer,1,0,0,0,2
34,4,Créez vous-même vos propres raccourcis,0,0,0,1,2
24,4,Devenez un as de la recherche,0,0,0,1,2
32,4,On ne va pas y aller par quatre chemins,0,0,0,1,2


## Résumé de l'algorithme en fonctions

In [115]:
def content_based_filtering(user_data,content_data,user_id,critere,num_contents=5):
    """
    Algorithme de recommandation de contenu

    Donne la liste de contenu à recommander à un utilisateur en fonction de ses préférences

    Le modèle compare les préférences de l'utilisateur en terme de contenu aux différents contenus que l'utilisateur
    n'a pas encore consommé et détermine les 5 contenus les plus proches des préférences de l'utilisateur

    Paramètres:
    ----------
        user_data(DataFrame) : Les données de l'utilisateur (tous les contenus qu'il a consommé)
        content_data(DataFrame) : Les contenus présents dans le domaine dans lequel l'utilisateur consomme des contenus
        user_id(int) : L'id de l'utilisateur
        critere(str) : Feedback ou Contenu_Validé , critère par lequel on détermine les préférences de l'utilisateur
        num_contents(int) : Le nombre de contenus à proposer

    Output :
        contenus(list) : liste de contenus recommandés pour l'utilisateur

    """
    tree = KDTree(content_data.drop(columns=["nom_contenu"]))
    user_row = user_data.query("id_user=={}".format(user_id))
    dist,ind = tree.query(user_row.drop(columns=["id_user",critere]),k=num_contents)
    return content_data.iloc[ind.tolist()[0]].nom_contenu.tolist()

def collaborative_filtering(user_data,content_data,user_id,critere,num_similar_users=3,num_contents=5):
    """
    Algorithme de recommandation de contenu

    Donne la liste de contenu à recommander à un utilisateur en fonction de ses préférences

    Le modèle détermine les utilisateurs les plus similaires à notre utilisateur, fait une moyenne de leur préférence pour avoir
    un profil de voisin type puis compare les préférences de ce voisin en terme de contenu aux différents contenus que l'utilisateur
    n'a pas encore consommé et détermine les 5 contenus les plus proches des préférences de l'utilisateur

    Paramètres
    ----------
        user_data(DataFrame) : Les données de l'utilisateur (tous les contenus qu'il a consommé)
        content_data(DataFrame) : Les contenus présents dans le domaine dans lequel l'utilisateur consomme des contenus
        user_id(int) : L'id de l'utilisateur
        critere(str) : Feedback ou Contenu_Validé , critère par lequel on détermine les préférences de l'utilisateur
        num_similar_users(int) : nombre de voisins semblables à notre utilisateur 
        num_contents(int): Le nombre de contenus à proposer
    
    Output:
    ------
        contenus(list) : liste des contenus à  recommander à notre utilisateur    
    
    """
    tree1 = KDTree(user_data.query("{}==True".format(critere)).drop(columns=["id_user",critere]))
    user_row = user_data.query("id_user=={}".format(user_id))
    dist1,ind1 = tree1.query(user_row.drop(columns=["id_user",critere]),k=num_similar_users)
    original_indices = list(user_data.index)
    filtered_indices = list(user_data.query("{}==True".format(critere)).index)
    mapping = {original_indices[i]:filtered_indices[i]  for i in range(min(len(filtered_indices), len(original_indices)))}
    original_indices = [mapping[i] for i in ind1.tolist()[0]]
    neighbour = pd.DataFrame([neighbour_profile(user_data.iloc[original_indices].query("id_user!={}".format(user_id)),critere=critere)])
    tree2 = KDTree(content_data.drop(columns=["nom_contenu"]))
    dist2,ind2 = tree2.query(neighbour[0:1],k=num_contents)
    return content_data.iloc[ind2.tolist()[0]].nom_contenu.tolist()

def content_recommandation(data,formation,domaine,user_id,critere):
    """
    Détermine l'algorithme de recommandation de contenu à utiliser en fonction de la validation du contenu ou du feedback

    Si notre utilisateur n'a pas de préférences définies (aucun contenu validé ou aucun bon feedback), on utilise le modèle de collaborative filtering
    Dans le cas contraire, on utilise le content based filtering

    
    Paramètres
    ----------
        data : Les données
        formation : La formation que souhaite suivre l'utilisateur
        domaine : le domaine dans lequel l'utilisateur veut consommer des contenus
        user_id : L'id de l'utilisateur
        critere : Feedback ou Contenu_Validé , critère par lequel on détermine les préférences de l'utilisateur

    Output:
    ------
        contenus(list) : liste des contenus à  recommander à notre utilisateur 

    """
    filtered_data= filtrage(data,formation,domaine)
    contents = content_features(filtered_data)
    contents = contents.loc[~contents["nom_contenu"].isin(list(content_features(filtered_data.query("id_user=={}".format(user_id))["nom_contenu"])))]   
    user_profiles = pd.DataFrame([profile(filtered_data,user,critere) for user in list(filtered_data.id_user.unique())])
    elements = {"Contenu_validé":"A_validé_contenus","Feedback":"A_bons_feedbacks"}
    column = ""
    for key,value in elements.items():
        if key == critere :
            column = value
    user_profiles = user_profiles[['id_user','Niveau_contenu','type_contenu_article','type_contenu_video','type_contenu_audio','type_contenu_rich_media','durée_contenu_en_minutes',column]]
    if user_profiles.query("id_user=={}".format(user_id)).to_dict("records")[0][column]==False:
        return collaborative_filtering(user_data = user_profiles,content_data=contents,user_id=user_id,critere=column)
    else :
        return content_based_filtering(user_data = user_profiles,content_data=contents,user_id=user_id,critere=column)

In [118]:
# En fonction du contenu validé
formation="DIGITAL"
domaine ="Information et données"
user_id = 1170
content_recommandation(data,formation,domaine,user_id,"Contenu_validé")

['Ne restez pas KO face à ces unités',
 'Devenez un as de la recherche',
 'On ne va pas y aller par quatre chemins',
 'Vos données dans le Cloud',
 'Zip Zap Zoup']

In [116]:
# En fonction du contenu validé
formation="DIGITAL"
domaine ="Information et données"
user_id = 60
content_recommandation(data,formation,domaine,user_id,"Contenu_validé")

["La France va donner l'Alsace ?!",
 'Supprimer sans paniquer',
 'On ne va pas y aller par quatre chemins',
 'Devenez un as de la recherche',
 'Créez vous-même vos propres raccourcis']

In [117]:
# En fonction du feedback
formation="DIGITAL"
domaine ="Information et données"
user_id = 60
content_recommandation(data,formation,domaine,user_id,"Feedback")

["La France va donner l'Alsace ?!",
 'Supprimer sans paniquer',
 'Créez vous-même vos propres raccourcis',
 'Devenez un as de la recherche',
 'On ne va pas y aller par quatre chemins']